# [0] - MLflow (exercises)

> **Dataset:** [Wine Quality](https://archive.ics.uci.edu/dataset/186/wine+quality) (UCI, CC BY 4.0) — **Tool:** [MLflow](https://mlflow.org/docs/latest/)
>
> The solutions are in `appliedai_practical_day0_mlflow_complete.ipynb`, and under each exercise in a collapsed block.

# Introduction

A model that lives in a notebook is not yet something anyone else can use. To use it, someone has to know which data and which settings produced it, how good it was, what to feed it, which version is the current one, and how to call it from their own code. The lifecycle diagram in the slides has a box for each of these questions. MLflow answers some of them, and in this practical you use it for those: tracking runs, packaging a model, a registry, and serving.

The task itself is deliberately small, a classifier on a table of wines, so that all the time goes to what happens around the model rather than to the model. Everything runs inside this Colab VM: the tracking server, its database, and later the model server. Nothing leaves the machine, and everything is gone when the runtime is. The notebook also runs on your own computer, in Jupyter or VS Code: use a fresh virtual environment with Python 3.10 or later, and the few steps that only make sense on Colab are skipped.

Each exercise has a difficulty and an importance rating out of 5, and a time estimate. If you are stuck for longer than the estimate, open the solution and move on. Sections 8️⃣ and 9️⃣ are optional.

## Content & Learning Objectives

### 1️⃣ Data

You load the wine-quality table and split it.

> ##### Learning Objectives
>
> * See why accuracy is a poor guide when one class is four times rarer than the other

### 2️⃣ A pyfunc model

You write the class that turns a raw row into a prediction.

> ##### Learning Objectives
>
> * Understand what MLflow stores when it stores a model
> * Put preprocessing and the decision threshold inside the model, not around it

### 3️⃣ A first run

You log a run with its parameters, metrics and model.

> ##### Learning Objectives
>
> * Decide what to log so that someone else can find, understand and reuse the run without you
> * Know what a model signature is, and what an input example is for

### 4️⃣ What goes in, what comes out

You feed the logged model broken rows.

> ##### Learning Objectives
>
> * See what the signature checks, what it lets through, and where the other checks belong

### 5️⃣ Experiments

You wrap training and logging in one function and compare runs.

> ##### Learning Objectives
>
> * Understand why comparisons need every run logged the same way
> * Read a table of runs: which differences are real, which metric decides

### 6️⃣ Registry

You register a model, promote another one, and roll back.

> ##### Learning Objectives
>
> * Use names, versions and aliases, so that code refers to a model without naming a run

### 7️⃣ Serve

You put the registered model behind an HTTP endpoint and call it.

> ##### Learning Objectives
>
> * See what changes for a caller who only sees JSON, and what does not

### 8️⃣ Promotion by rule (optional)

You run a small sweep, measure how much a score moves with the split alone, and write the function that decides whether a new model may become the champion.

> ##### Learning Objectives
>
> * Separate a real improvement from the noise of the split
> * Write down the metric and the margin before looking at the results

### 9️⃣ After deployment (optional)

A model trained on white wines starts receiving red ones. You see what the metrics hide, and write a check that notices from the inputs alone.

> ##### Learning Objectives
>
> * See why accuracy can stay high while a model fails
> * Detect a change in the inputs without labels

## Setup code

Install MLflow, start a tracking server in the background, and open its UI.

In [ ]:
# MLflow version pinned to the one this practical was tested with.
%pip install -q mlflow==3.18.0

In [ ]:
import atexit
import io
import logging
import os
import socket
import subprocess
import sys
import time
import urllib.request
import warnings
import zipfile
from pathlib import Path

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "true"   # its run links point to 127.0.0.1: use the UI link below
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"
warnings.filterwarnings("ignore", module="mlflow")
for noisy in ["mlflow.pyfunc", "mlflow.utils.environment", "mlflow.tracking.fluent", "mlflow.tracking._model_registry.fluent",
              "mlflow.store.model_registry.abstract_store"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)

import mlflow
import mlflow.pyfunc
import pandas as pd
import requests
from mlflow.models import infer_signature

IN_COLAB = "google.colab" in sys.modules   # Colab-only steps (the proxy and the link to the UI) check this


def free_port():
    """A port nobody is using (on a Mac, 5000 is often taken by AirPlay)."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

The next cell defines the tests for the exercises. Run it; you do not need to read it.

In [ ]:
#@title Tests for the exercises
"""Tests for the MLflow lab (Day 0).

The build pastes this file into a collapsed cell at the top of the notebook, so the test
functions are defined there and the exercise cells call them directly.

Each test takes what you wrote, checks it, and either prints that it passed or stops
with an AssertionError that says what is wrong. The tests use their own small made-up
wine table, so they do not depend on your train/test split.
"""
import contextlib
import io

import mlflow
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

COLUMNS = ["type", "fixed acidity", "volatile acidity", "citric acid", "residual sugar", "chlorides",
           "free sulfur dioxide", "total sulfur dioxide", "density", "pH", "sulphates", "alcohol"]
METRICS = {"accuracy", "balanced_accuracy", "precision", "recall", "roc_auc"}


def _wines(n=200, seed=0):
    """A made-up table with the columns of the real one, and a label that depends on alcohol."""
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, len(COLUMNS) - 1)), columns=COLUMNS[1:])
    X["alcohol"] = rng.normal(10.5, 1.2, n)
    X.insert(0, "type", rng.choice(["red", "white"], n))
    y = pd.Series((X["alcohol"] + rng.normal(0, 0.8, n) > 11.2).astype(int), name="good")
    return X, y


def _passed(name):
    print(f"All tests in `{name}` passed!")


# ---------------------------------------------------------------- Exercise 1: the pyfunc model

def test_features(WineModel):
    X, _ = _wines(20)
    before = X.copy()
    out = WineModel._features(X)
    assert out is not None, "_features returned None: it must return a DataFrame."
    assert isinstance(out, pd.DataFrame), f"_features must return a DataFrame, not {type(out).__name__}."
    assert X.equals(before), "_features changed its input. Work on a copy (df.copy())."
    assert list(out.columns) == list(X.columns), "_features must keep the same columns, in the same order."
    assert pd.api.types.is_numeric_dtype(out["type"]), "`type` is still text: turn red/white into numbers."
    assert set(out["type"].unique()) <= {0, 1}, "Encode `type` as 0 and 1."
    red = (X["type"] == "red").to_numpy()
    assert (out["type"].to_numpy()[red] == 1).all() and (out["type"].to_numpy()[~red] == 0).all(), \
        "Use red = 1, white = 0, so that everyone's models agree."
    assert out.drop(columns="type").equals(X.drop(columns="type")), "_features must leave the other columns as they are."
    _passed(f"{WineModel.__name__}._features")


def test_fit(WineModel):
    X, y = _wines()
    model = WineModel(LogisticRegression(max_iter=500))
    try:
        returned = model.fit(X, y)
    except ValueError as e:
        raise AssertionError(f"fit failed on raw rows ({str(e)[:80]}...). Did you pass the rows through _features?") from None
    assert returned is model, "fit must return self, so that you can write WineModel(...).fit(X, y)."
    assert hasattr(model.estimator, "coef_"), "fit returned, but the estimator inside was not fitted."
    _passed(f"{WineModel.__name__}.fit")


def test_predict(WineModel):
    X, y = _wines()
    model = WineModel(LogisticRegression(max_iter=500), threshold=0.5).fit(X, y)
    try:
        out = model.predict(None, X)
    except ValueError as e:
        raise AssertionError(f"predict failed on raw rows ({str(e)[:80]}...). Did you pass the rows through _features?") from None
    assert out is not None, "predict returned None: it must return a DataFrame."
    assert isinstance(out, pd.DataFrame), f"predict must return a DataFrame, not {type(out).__name__}."
    assert sorted(out.columns) == ["good", "p_good"], f"predict must return exactly the columns `good` and `p_good`, not {list(out.columns)}."
    assert len(out) == len(X), f"One output row per input row: got {len(out)} for {len(X)}."
    p = out["p_good"].to_numpy()
    assert ((p >= 0) & (p <= 1)).all(), "`p_good` must be a probability, between 0 and 1."
    expected = model.estimator.predict_proba(WineModel._features(X))[:, 1]
    assert np.allclose(p, expected), "`p_good` must be the estimator's probability of class 1 (predict_proba(...)[:, 1])."
    assert set(out["good"].unique()) <= {0, 1}, "`good` must be 0 or 1."
    assert (out["good"].to_numpy() == (p >= 0.5)).all(), "`good` must be 1 exactly when p_good >= threshold."
    model.threshold = 0.0
    assert (model.predict(None, X)["good"] == 1).all(), "With threshold 0 every wine is good: use self.threshold, not a fixed 0.5."
    model.threshold = 1.01
    assert (model.predict(None, X)["good"] == 0).all(), "With threshold above 1 no wine is good: use self.threshold."
    _passed(f"{WineModel.__name__}.predict")


# ---------------------------------------------------------------- Exercises 2 and 3: logging runs

def _check_logged(run_id, features, params_expected=()):
    client = mlflow.MlflowClient()
    data = client.get_run(run_id).data
    assert data.params, "Nothing was logged as a parameter: use mlflow.log_params."
    for key in ("threshold", "features", *params_expected):
        assert key in data.params, f"Parameter `{key}` is missing. Logged: {sorted(data.params)}."
    for f in features:
        assert f in data.params["features"], f"The `features` parameter does not list `{f}`."
    missing = METRICS - set(data.metrics)
    assert not missing, f"Metrics missing: {sorted(missing)}. Use mlflow.log_metrics(m)."
    logged = [a.path for a in client.list_artifacts(run_id)]
    models = mlflow.search_logged_models(filter_string=f"source_run_id = '{run_id}'", output_format="list")
    assert models or "model" in logged, "No model was logged in this run: use mlflow.pyfunc.log_model(name='model', ...)."
    loaded = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
    sig = loaded.metadata.signature
    assert sig is not None, "The model has no signature: pass signature=infer_signature(inputs, outputs)."
    assert sig.inputs.input_names() == list(features), \
        f"The signature's inputs should be the training columns {list(features)}, not {sig.inputs.input_names()}."
    assert sig.outputs is not None and sorted(sig.outputs.input_names()) == ["good", "p_good"], \
        "The signature's outputs should be `good` and `p_good`: infer it from the output of predict."
    assert loaded.metadata.saved_input_example_info is not None, "No input example: pass input_example=a few training rows."
    return client, data, loaded


def test_first_run(run_id, X_train):
    _, data, loaded = _check_logged(run_id, list(X_train.columns), params_expected=("n_estimators",))
    out = loaded.predict(X_train.head(5))
    assert sorted(out.columns) == ["good", "p_good"], "The logged model does not return `good` and `p_good`."
    _passed("test_first_run")


def test_run_experiment(run_experiment):
    features = ["type", "alcohol"]
    result = run_experiment(LogisticRegression(max_iter=500), features=features, threshold=0.4, run_name="test-run_experiment")
    assert result is not None and len(result) == 2, "run_experiment must return (run_id, metrics)."
    run_id, m = result
    try:
        assert isinstance(m, dict) and METRICS <= set(m), "run_experiment must return the metrics dict as its second value."
        client, data, _ = _check_logged(run_id, features, params_expected=("estimator", "max_iter"))
        assert data.params["estimator"] == "LogisticRegression", \
            f"The `estimator` parameter should be the class name, LogisticRegression, not {data.params['estimator']!r}."
        assert float(data.params["threshold"]) == 0.4, "Log the threshold you were given, not a fixed one."
        assert "density" not in data.params["features"], "Log only the features you trained on."
    finally:
        mlflow.MlflowClient().delete_run(run_id)   # keep the test run out of your table
    _passed(run_experiment.__name__)


# ---------------------------------------------------------------- Exercise 4: the registry

def test_champion(name, expected_run_id):
    client = mlflow.MlflowClient()
    try:
        mv = client.get_model_version_by_alias(name, "champion")
    except mlflow.exceptions.MlflowException:
        raise AssertionError(f"`{name}` has no version with the alias `champion`.") from None
    assert mv.run_id == expected_run_id, \
        f"`champion` points to version {mv.version}, which comes from run {mv.run_id[:8]}, not from {expected_run_id[:8]}."
    loaded = mlflow.pyfunc.load_model(f"models:/{name}@champion")
    assert loaded.metadata.run_id == expected_run_id, "Loading models:/...@champion does not give the expected run."
    _passed("test_champion")


# ---------------------------------------------------------------- Going further A: promotion by rule

class _Constant(mlflow.pyfunc.PythonModel):
    """A stand-in model for the promotion test: returns its first input column."""

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        return model_input.iloc[:, :1]


def test_promote_if_better(promote_if_better):
    client = mlflow.MlflowClient()
    name = "test-promotion"
    example = pd.DataFrame({"x": [1.0, 2.0]})
    run_ids = {}
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):   # MLflow's messages while the test sets itself up
            for label, score in [("champion", 0.70), ("small_gain", 0.72), ("large_gain", 0.80)]:
                with mlflow.start_run(run_name=f"test-promotion-{label}") as run:
                    mlflow.log_metric("balanced_accuracy", score)
                    mlflow.pyfunc.log_model(name="model", python_model=_Constant(), input_example=example)
                run_ids[label] = run.info.run_id
            first = mlflow.register_model(f"runs:/{run_ids['champion']}/model", name)
            client.set_registered_model_alias(name, "champion", first.version)

        result = promote_if_better(run_ids["small_gain"], "balanced_accuracy", min_gain=0.05, name=name)
        assert result is not None, "promote_if_better returned None: return True when it promotes, False when it does not."
        assert result is False, "A gain of 0.02 with min_gain=0.05 must not be promoted."
        assert client.get_model_version_by_alias(name, "champion").run_id == run_ids["champion"], \
            "The champion moved although the gain was below min_gain."

        result = promote_if_better(run_ids["large_gain"], "balanced_accuracy", min_gain=0.05, name=name)
        assert result is True, "A gain of 0.10 with min_gain=0.05 must be promoted (return True)."
        new = client.get_model_version_by_alias(name, "champion")
        assert new.run_id == run_ids["large_gain"], "After promotion, `champion` must point to a version from the candidate run."
        assert new.tags, "Tag the promoted version with the reason (client.set_model_version_tag), so the registry keeps a record."
    finally:
        try:
            client.delete_registered_model(name)
        except mlflow.exceptions.MlflowException:
            pass
        for run_id in run_ids.values():
            client.delete_run(run_id)
    _passed(promote_if_better.__name__)


# ---------------------------------------------------------------- Going further B: drift

def test_drift_report(drift_report):
    rng = np.random.default_rng(1)
    reference = pd.DataFrame({"type": ["white"] * 500, "a": rng.normal(0, 1, 500), "b": rng.normal(5, 2, 500)})
    incoming = pd.DataFrame({"type": ["red"] * 300, "a": rng.normal(0, 1, 300), "b": rng.normal(1, 2, 300)})
    report = drift_report(reference, incoming)
    assert report is not None, "drift_report returned None: it must return a pandas Series."
    assert isinstance(report, pd.Series), f"drift_report must return a pandas Series, not {type(report).__name__}."
    assert set(report.index) == {"a", "b"}, f"One value per numeric column (not `type`): got {list(report.index)}."
    assert abs(report["b"] - (-2.0)) < 0.3, \
        f"`b` moved by about -2 reference standard deviations; you returned {report['b']:.2f}. Use (incoming mean - reference mean) / reference std."
    assert abs(report["a"]) < 0.3, f"`a` did not move, but you report {report['a']:.2f}."
    assert list(report.index) == ["b", "a"], "Sort the columns by the size of the shift, largest first."
    _passed(drift_report.__name__)

The tracking server is a small web application with a database behind it. Your code sends it parameters, metrics and files; the UI reads them back. Here it runs in the background of this VM, with its records in `mlflow.db` and the files (models, plots) in `./mlartifacts`. In a team it would run on a shared machine, and everyone would point `set_tracking_uri` at it.

Runs are grouped into **experiments**: one experiment per question you are working on, here `wine`. The same server also hosts the **model registry**, which you use in section 6️⃣.

The server takes one to three minutes to start on Colab, a few seconds on a laptop.

In [ ]:
PORT = free_port()
TRACKING_URI = f"http://127.0.0.1:{PORT}"

command = [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(PORT),
           "--backend-store-uri", "sqlite:///mlflow.db", "--artifacts-destination", "./mlartifacts"]
if IN_COLAB:
    # Colab's proxy calls the server with the VM's internal name, which MLflow refuses by default.
    command += ["--allowed-hosts", "*.codatalab-user-runtimes.internal:*,localhost:*,127.0.0.1:*",
                "--cors-allowed-origins", "https://*.colab.dev"]

server = subprocess.Popen(command, stdout=open("mlflow_server.log", "w"), stderr=subprocess.STDOUT)
atexit.register(server.terminate)   # stop it when the notebook's kernel stops

for _ in range(300):
    if server.poll() is not None:
        raise RuntimeError("the server stopped, see mlflow_server.log")
    try:
        if requests.get(f"{TRACKING_URI}/health").ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("server did not start, see mlflow_server.log")

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("wine")

The next cell shows a link to the MLflow UI. Open it in a new tab and keep it open. In the UI, switch the toggle at the top left from *GenAI* to *Model training*.

In [ ]:
from IPython.display import Javascript, display

# The browser asks Colab for the address of the port and draws the link; Python does not wait for it.
LINK = """
(async () => {
  const url = await google.colab.kernel.proxyPort(%d);
  const a = document.createElement("a");
  a.href = url; a.target = "_blank"; a.textContent = "Open the MLflow UI";
  document.body.appendChild(a);
})();
"""
if IN_COLAB:
    display(Javascript(LINK % PORT))
else:
    print("MLflow UI:", TRACKING_URI)

# 1️⃣ Data

> ##### Learning Objectives
>
> * See why accuracy is a poor guide when one class is four times rarer than the other

Wine quality (UCI): red and white wines, physico-chemical measurements, a quality score from 3 to 9. The target is `good` = quality >= 7.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/186/wine+quality.zip"
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(DATA_URL).read()))

frames = []
for color in ["red", "white"]:
    df = pd.read_csv(z.open(f"winequality-{color}.csv"), sep=";")
    df.insert(0, "type", color)
    frames.append(df)

wine = pd.concat(frames, ignore_index=True)
wine["good"] = (wine["quality"] >= 7).astype(int)
wine = wine.drop(columns="quality")

print(wine.shape, "positives:", wine["good"].mean().round(3))
wine.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = wine.drop(columns="good")
y = wine["good"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

About one wine in five is good. A model that always says "not good" is right four times out of five, so accuracy alone will flatter every model: read balanced accuracy and ROC AUC next to it.

The threshold of 7 is a choice, not a fact about wine. So is the split above (30% held out, seed 0). Both change every number you will see, which is a first hint of what has to be written down next to a result.

# 2️⃣ A pyfunc model

> ##### Learning Objectives
>
> * Understand what MLflow stores when it stores a model
> * Put preprocessing and the decision threshold inside the model, not around it

MLflow can save a scikit-learn estimator as it is. But the estimator is not the whole model: something has to turn `type` into a number before calling it, and something has to pick the threshold that turns a probability into a yes or a no. If that code stays in the notebook, whoever loads the model has to write it again, and will write it slightly differently.

A `pyfunc` model is a Python class with a `predict(context, model_input, params)` method. MLflow pickles the object and stores it with the list of packages it needs. From then on every consumer calls the same `predict`: `mlflow.pyfunc.load_model(...)` in Python, or the REST server in section 7️⃣. Whatever happens between a raw row and the prediction lives inside the class and is logged with it.

Two things to know about this format:

* MLflow saves the object with `cloudpickle`. A pickled object only loads back in a Python with compatible versions of the same packages, which is why MLflow writes a `requirements.txt` next to the model.
* `context` gives `predict` access to files saved with the model (a vocabulary, a lookup table). This model has none, so `context` is unused here.

### Exercise - implement `WineModel`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 15 minutes on this exercise.
> ```

Fill in the three methods. `fit` and `predict` are a few lines each; the point is where the encoding and the threshold live, not the code. `predict` gets `context` and `params` from MLflow: you can ignore both here.

In [ ]:
class WineModel(mlflow.pyfunc.PythonModel):

    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator   # any sklearn classifier with predict_proba
        self.threshold = threshold   # p_good at or above this means good

    @staticmethod
    def _features(df: pd.DataFrame) -> pd.DataFrame:
        """Raw rows -> what the estimator takes.

        `type` is "red" or "white": make it red = 1, white = 0 (when the column is there: in section 5
        you will train on fewer columns). Return a new DataFrame and leave `df` unchanged.
        """
        raise NotImplementedError()

    def fit(self, X: pd.DataFrame, y) -> "WineModel":
        """Fit the estimator on the encoded rows. Return self."""
        raise NotImplementedError()

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        """Raw rows -> a DataFrame with two columns:
        `p_good`, the estimator's probability of class 1, and `good`, 1 when p_good >= threshold, else 0.
        """
        raise NotImplementedError()


test_features(WineModel)
test_fit(WineModel)
test_predict(WineModel)

<details>
<summary>Solution</summary>

```python
class WineModel(mlflow.pyfunc.PythonModel):

    def __init__(self, estimator, threshold=0.5):
        self.estimator = estimator   # any sklearn classifier with predict_proba
        self.threshold = threshold   # p_good at or above this means good

    @staticmethod
    def _features(df: pd.DataFrame) -> pd.DataFrame:
        """Raw rows -> what the estimator takes.

        `type` is "red" or "white": make it red = 1, white = 0 (when the column is there: in section 5
        you will train on fewer columns). Return a new DataFrame and leave `df` unchanged.
        """
        df = df.copy()
        if "type" in df:
            df["type"] = (df["type"] == "red").astype(int)
        return df

    def fit(self, X: pd.DataFrame, y) -> "WineModel":
        """Fit the estimator on the encoded rows. Return self."""
        self.estimator.fit(self._features(X), y)
        return self

    def predict(self, context, model_input: pd.DataFrame, params=None) -> pd.DataFrame:
        """Raw rows -> a DataFrame with two columns:
        `p_good`, the estimator's probability of class 1, and `good`, 1 when p_good >= threshold, else 0.
        """
        p = self.estimator.predict_proba(self._features(model_input))[:, 1]
        return pd.DataFrame({"good": (p >= self.threshold).astype(int), "p_good": p})
```
</details>

The metrics every run reports:

In [ ]:
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, roc_auc_score)


def metrics(y_true, out):
    return {
        "accuracy": accuracy_score(y_true, out["good"]),
        "balanced_accuracy": balanced_accuracy_score(y_true, out["good"]),
        "precision": precision_score(y_true, out["good"], zero_division=0),
        "recall": recall_score(y_true, out["good"]),
        "roc_auc": roc_auc_score(y_true, out["p_good"]),
    }

# 3️⃣ A first run

> ##### Learning Objectives
>
> * Decide what to log so that someone else can find, understand and reuse the run without you
> * Know what a model signature is, and what an input example is for

A run is one record on the tracking server. It holds **parameters** (what you chose), **metrics** (the numbers that came out) and **artifacts** (files, the model among them). There are also **tags**: free labels such as the data source or who ran it, which you can search and filter on. Parameters are fixed once logged; metrics can be logged many times (one value per epoch, for example) and the UI plots them.

The question to ask when deciding what to log: if someone finds this run in six months, can they tell what was done, and use the model without asking you?

Two things go with the model:

* the **signature**, `infer_signature(inputs, outputs)`: the names and types of the columns that go in and come out. MLflow checks every call against it.
* an **input example**: a few real rows stored next to the model, so that whoever picks it up sees what a valid input looks like.

### Exercise - log the run

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
>
> You should spend up to 15 minutes on this exercise.
> ```

Training and scoring are given. Log, inside the `with` block:

* parameters, with `mlflow.log_params(dict)`: the estimator's, and also the threshold and the feature list, which change the model as much as `n_estimators` does
* the metrics, with `mlflow.log_metrics(dict)`
* the model, with `mlflow.pyfunc.log_model(name="model", python_model=..., input_example=..., signature=...)`

Anything else you would want in six months? The data's URL, the split seed, the number of rows, who ran it. Add it with `mlflow.set_tag(key, value)` or `mlflow.set_tags(dict)`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

params = {"n_estimators": 100, "max_depth": None, "random_state": 0}

with mlflow.start_run(run_name="rf-baseline") as run:
    model = WineModel(RandomForestClassifier(**params)).fit(X_train, y_train)
    out = model.predict(None, X_test)
    m = metrics(y_test, out)

    raise NotImplementedError()

print(run.info.run_id)
test_first_run(run.info.run_id, X_train)
m

<details>
<summary>Solution</summary>

```python
from sklearn.ensemble import RandomForestClassifier

params = {"n_estimators": 100, "max_depth": None, "random_state": 0}

with mlflow.start_run(run_name="rf-baseline") as run:
    model = WineModel(RandomForestClassifier(**params)).fit(X_train, y_train)
    out = model.predict(None, X_test)
    m = metrics(y_test, out)

    mlflow.log_params({**params, "threshold": model.threshold, "features": list(X_train.columns)})
    mlflow.log_metrics(m)
    mlflow.set_tags({"data": DATA_URL, "split_seed": 0, "n_train": len(X_train), "n_test": len(X_test)})
    mlflow.pyfunc.log_model(
        name="model",
        python_model=model,
        input_example=X_train.head(3),
        signature=infer_signature(X_train, out),
    )

print(run.info.run_id)
m
```
</details>

You should find a balanced accuracy of about 0.77 and a ROC AUC of about 0.92.

Now load the model back from the server, as anyone else would: by the run's id, without `WineModel` or `X_train` from this notebook's code.

In [ ]:
loaded = mlflow.pyfunc.load_model(f"runs:/{run.info.run_id}/model")
print(loaded.metadata.signature)
loaded.predict(X_test.head())

In the UI (*Model training* mode): open the `wine` experiment and the run. Find its parameters, metrics and tags. The model has a page of its own: click `model` next to any metric, or open the experiment's *Models* tab. Its *Artifacts* tab holds the `MLmodel` file (how to load it, the signature), `requirements.txt` (the packages it needs) and `input_example.json`.

# 4️⃣ What goes in, what comes out

> ##### Learning Objectives
>
> * See what the signature checks, what it lets through, and where the other checks belong

The signature is a contract between the model and whoever calls it. Try to break it.

In production the rows do not come from your own `X_test`. They come from someone else's code, a form, another team's database. Below are six ways a single row can arrive wrong. For each one, decide first whether the model should refuse it, then run the cell.

In [ ]:
row = X_test.head(1)

cases = {
    "missing column": row.drop(columns="alcohol"),
    "wrong type": row.assign(alcohol="12.5"),
    "extra column": row.assign(vintage=2019),
    "missing value": row.assign(alcohol=None),
    "unknown category": row.assign(type="rosé"),
    "values out of range": row.assign(alcohol=90.0, pH=-3.0),
}

for name, df in cases.items():
    try:
        result = loaded.predict(df).to_dict(orient="records")
    except Exception as e:
        result = f"{type(e).__name__}: {str(e).rsplit('Error:', 1)[-1].strip()[:200]}"
    print(f"{name:22s} -> {result}")

You should find that only the missing column is refused. Add your own cases.

<details>
<summary>Question - which of these should have been refused, and where does each check belong?</summary>

| case | what happened | should it? | where the check belongs |
|---|---|---|---|
| missing column | refused by the signature | yes | signature |
| wrong type, `"12.5"` as text | a prediction: the text is read as 12.5 | arguable | signature, or the caller |
| extra column | ignored, with a warning | fine | signature |
| missing value | a prediction | no | `predict` |
| unknown category, `rosé` | a prediction | no | `_features` |
| values out of range | a prediction | no | `predict`, or the caller |

These are the results on Colab (pandas 2.2). With pandas 3 the text `"12.5"` is refused instead: the same model and the same signature, a different answer, because a library underneath changed.

The signature checks names and types, and that required columns are present. It does not check that a column has a value, which values a text column may take, or what is physically possible.

* *Missing value*: the random forest accepts NaN (scikit-learn 1.4 and later) and answers. Refuse the row in `predict`, or fill it in on purpose; either way, decide.
* *Rosé*: `_features` only asks "is it red?", so a rosé is scored as a white. Only the model knows its categories: list them in `_features` and refuse the others.
* *90% alcohol, pH −3*: a valid float, an impossible wine, and the model still gives a probability. Ranges are domain knowledge. A check in `predict` travels with the model; a check in the caller has to be written again by every caller.
</details>

# 5️⃣ Experiments

> ##### Learning Objectives
>
> * Understand why comparisons need every run logged the same way
> * Read a table of runs: which differences are real, which metric decides

One run is a record; many runs are a comparison. Every run below goes into the same experiment, `wine`, and the UI (or `mlflow.search_runs`) puts them side by side. This only works if every run logs the same things in the same way, which is why it is worth a function.

### Exercise - implement `run_experiment`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

The same as your first run, on `X_train[features]` and `X_test[features]`, with the given threshold. Log `estimator.get_params()`, the estimator's class name as `estimator`, the threshold and the feature list. Infer the signature and take the input example on the selected features only. The test runs your function once and deletes the run it made.

In [ ]:
def run_experiment(estimator, features=None, threshold=0.5, run_name=None, tags=None):
    """Trains, scores and logs one run. Returns (run_id, metrics)."""
    features = features or list(X_train.columns)
    with mlflow.start_run(run_name=run_name, tags=tags) as run:
        raise NotImplementedError()
    return run.info.run_id, m


test_run_experiment(run_experiment)

<details>
<summary>Solution</summary>

```python
def run_experiment(estimator, features=None, threshold=0.5, run_name=None, tags=None):
    """Trains, scores and logs one run. Returns (run_id, metrics)."""
    features = features or list(X_train.columns)
    with mlflow.start_run(run_name=run_name, tags=tags) as run:
        model = WineModel(estimator, threshold).fit(X_train[features], y_train)
        out = model.predict(None, X_test[features])
        m = metrics(y_test, out)

        mlflow.log_params({**estimator.get_params(), "estimator": type(estimator).__name__,
                           "threshold": threshold, "features": features})
        mlflow.log_metrics(m)
        mlflow.pyfunc.log_model(
            name="model",
            python_model=model,
            input_example=X_train[features].head(3),
            signature=infer_signature(X_train[features], out),
        )
    return run.info.run_id, m
```
</details>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

run_experiment(RandomForestClassifier(n_estimators=300, max_depth=8, random_state=0), run_name="rf-depth-8")
run_experiment(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), run_name="logreg")
run_experiment(RandomForestClassifier(n_estimators=100, random_state=0), threshold=0.3, run_name="rf-threshold-0.3")
run_experiment(RandomForestClassifier(n_estimators=100, random_state=0),
               features=["type", "alcohol", "volatile acidity", "density"], run_name="rf-4-features")

In [ ]:
# The same table as the UI, as a DataFrame
runs = mlflow.search_runs(order_by=["metrics.balanced_accuracy DESC"])
cols = ["run_id", "tags.mlflow.runName", "params.threshold", "metrics.accuracy", "metrics.balanced_accuracy",
        "metrics.precision", "metrics.recall", "metrics.roc_auc"]
runs[cols]

You should find `rf-threshold-0.3` at the top, with a balanced accuracy of about 0.83. In the UI, select the runs and press *Compare*.

<details>
<summary>Question - is <code>rf-threshold-0.3</code> the best model?</summary>

It is the best by balanced accuracy, and it loses to `rf-baseline` on precision. Sorting by one metric picks a winner for that metric; which metric matters depends on what a wrong answer costs. Section 8️⃣ comes back to this.

Also, the runs share one test set of about 2,000 wines, of which about 400 are good. A difference of 0.01 in recall is four wines. Before trusting a ranking, ask whether it would survive another split.
</details>

# 6️⃣ Registry

> ##### Learning Objectives
>
> * Use names, versions and aliases, so that code refers to a model without naming a run

One name, many versions, an alias that points to the one in use. Promotion and rollback are alias moves, not code changes.

Registering a run's model under a name (`wine-good`) gives it the next version number: 1, 2, ... An alias such as `champion` is a pointer to one version, and can be moved. Code that uses the model loads `models:/wine-good@champion` and never names a run or a version. (Older MLflow used fixed "stages", Staging and Production; aliases replace them.)

Why not just use the run id? A run id points to one training run and never changes. The application that uses the model should not change every time a better model is trained. With a name and an alias, the application keeps loading `models:/wine-good@champion`, and the decision of which version that is gets made, and recorded, in the registry.

In [ ]:
client = mlflow.MlflowClient()

chosen = runs.iloc[0]
mv = mlflow.register_model(f"runs:/{chosen.run_id}/model", "wine-good")
client.set_registered_model_alias("wine-good", "champion", mv.version)

champion = mlflow.pyfunc.load_model("models:/wine-good@champion")
print(champion.metadata.run_id == chosen.run_id)
champion.predict(X_test.head())

### Exercise - promote, then roll back

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

A colleague prefers the logistic regression: simpler, and easier to explain. Register its model as a new version of `wine-good` and point `champion` at it. Then compare its numbers with the champion's in the table above, and put the previous version back. You need `mlflow.register_model` and `client.set_registered_model_alias`. Watch the *Model registry* page of the UI as you go.

In [ ]:
logreg_id = runs.loc[runs["tags.mlflow.runName"] == "logreg", "run_id"].iloc[0]

raise NotImplementedError()

test_champion("wine-good", logreg_id)

<details>
<summary>Solution</summary>

```python
logreg_id = runs.loc[runs["tags.mlflow.runName"] == "logreg", "run_id"].iloc[0]

mv2 = mlflow.register_model(f"runs:/{logreg_id}/model", "wine-good")
client.set_registered_model_alias("wine-good", "champion", mv2.version)
```
</details>

Now point `champion` back at the version you registered first (`mv`).

In [ ]:
raise NotImplementedError()

test_champion("wine-good", chosen.run_id)

<details>
<summary>Solution</summary>

```python
client.set_registered_model_alias("wine-good", "champion", mv.version)
```
</details>

<details>
<summary>Question - what changed when you promoted the logistic regression, and what did not?</summary>

Nothing was retrained and no code was changed: the only thing that moved is a pointer. Whatever loads `models:/wine-good@champion` gets the version it points to at load time, and the registry keeps every version, so going back is one more move of the same pointer.
</details>

# 7️⃣ Serve

> ##### Learning Objectives
>
> * See what changes for a caller who only sees JSON, and what does not

The same model behind an HTTP endpoint. The contract is the same, the payload is JSON.

This is how other programs use a model: a web app, a batch job, a service in another language. They send rows as JSON to `POST /invocations` and get predictions back. The rows can be sent in a few shapes; here we use `dataframe_records`, a list with one dictionary per row.

`mlflow models serve` loads the model once, when it starts, and answers `POST /invocations`. Moving the alias afterwards changes nothing until the server restarts. `--env-manager local` uses the packages already installed in this VM. On a real server you would let MLflow build the environment from the model's `requirements.txt` (`--env-manager virtualenv`), or package model and server in a Docker image (`mlflow models build-docker`).

The next cell starts the model server in the background; it takes up to a minute.

In [ ]:
SERVE_PORT = free_port()

serving = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "models", "serve", "-m", "models:/wine-good@champion", "-p", str(SERVE_PORT), "--env-manager", "local"],
    stdout=open("mlflow_serve.log", "w"), stderr=subprocess.STDOUT,
    env={**os.environ, "MLFLOW_TRACKING_URI": TRACKING_URI,
         "PATH": os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]},
)
atexit.register(serving.terminate)

for _ in range(120):
    if serving.poll() is not None:
        raise RuntimeError("the model server stopped, see mlflow_serve.log")
    try:
        if requests.get(f"http://127.0.0.1:{SERVE_PORT}/ping").ok:
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("model server did not start, see mlflow_serve.log")

In [ ]:
INVOCATIONS = f"http://127.0.0.1:{SERVE_PORT}/invocations"
payload = {"dataframe_records": X_test.head(3).to_dict(orient="records")}
r = requests.post(INVOCATIONS, json=payload)
print(r.status_code)
r.json()

### Exercise - send the broken rows over HTTP

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

Send each of the broken rows from section 4️⃣ to the endpoint, in the same format as above. Compare status codes and messages with the exceptions you got in Python.

In [ ]:
for name, df in cases.items():
    raise NotImplementedError()
    print(f"{name:22s} -> {r.status_code} {r.text[:160]}")

<details>
<summary>Solution</summary>

```python
for name, df in cases.items():
    r = requests.post(INVOCATIONS, json={"dataframe_records": df.to_dict(orient="records")})
    print(f"{name:22s} -> {r.status_code} {r.text[:160]}")
```
</details>

You should find status 400 for the missing column, and 200 for everything else.

<details>
<summary>Question - what does a caller who only sees the HTTP response learn?</summary>

* *missing column*: status 400, `INVALID_PARAMETER_VALUE`, and a long message that echoes the input rows back. Fine here; not with personal data in the rows.
* everything else, the text `"12.5"` included: status 200 and a prediction, as in Python. (With pandas 3, Python refuses `"12.5"` but the HTTP server still accepts it.)

A caller who sees status 200 and a number has no way to know the row was wrong. Whatever the model lets through in Python it lets through over HTTP, and the JSON layer adds conversions of its own. The checks have to live in the model.
</details>

### What you have, and what is missing

You now have runs that say what was done, a model that carries its own preprocessing and threshold, a contract for its inputs, a name with a current version, and an endpoint.

Look back at the lifecycle diagram from the slides. MLflow did not touch most of it: where the data came from and which version of it you used, whether the rows coming in next month look like the ones you trained on, when to retrain, and who decides that a new version may become the champion. Those need other tools, or people, and they are where most production problems start. The two optional sections below take a first look at the last two.

# 8️⃣ Promotion by rule (optional)

> ##### Learning Objectives
>
> * Separate a real improvement from the noise of the split
> * Write down the metric and the margin before looking at the results

In section 6️⃣ the champion moved because a colleague preferred another model. A team needs a rule instead, written down and applied the same way every time. A rule needs two things:

* **the metric that matters**, which is a decision about the business, not about the model;
* **a margin**: how much better a new model must be before the difference is real and not luck.

First, more candidates: a small sweep over tree depth and decision threshold, all logged with your `run_experiment` and tagged so you can find them in the UI.

In [ ]:
sweep = []
for depth in [None, 8]:
    for threshold in [0.3, 0.4, 0.5]:
        estimator = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=0)
        run_id, m = run_experiment(estimator, threshold=threshold, run_name=f"sweep-depth-{depth}-threshold-{threshold}",
                                   tags={"sweep": "A"})
        sweep.append({"run_id": run_id, "max_depth": depth, "threshold": threshold, **m})
sweep = pd.DataFrame(sweep).sort_values("balanced_accuracy", ascending=False)
sweep.drop(columns="run_id").round(3)

How much would these numbers move if we had split the data differently? Train the same model on five different splits and look at the spread. Any difference between two runs smaller than this spread is not evidence that one model is better.

In [ ]:
split_scores = []
for seed in range(5):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    model = WineModel(RandomForestClassifier(n_estimators=100, random_state=0), threshold=0.3).fit(Xa, ya)
    split_scores.append(metrics(yb, model.predict(None, Xb))["balanced_accuracy"])
split_scores = pd.Series(split_scores)
print("balanced accuracy on five splits:", split_scores.round(3).tolist())
print("standard deviation:", round(split_scores.std(), 4))

You should find a standard deviation of about 0.011.

### Exercise - implement `promote_if_better`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 15 minutes on this exercise.
> ```

Read the candidate's metric and the current champion's from the tracking server, both by run id. If the candidate is better by more than `min_gain`, register it, move `champion` to the new version, and tag that version with the reason. Return whether it was promoted.

The test uses its own registered model, `test-promotion`, and deletes it afterwards.

In [ ]:
def promote_if_better(candidate_run_id, metric="balanced_accuracy", min_gain=0.0, name="wine-good"):
    """Returns True if the candidate became the champion of `name`, False otherwise."""
    raise NotImplementedError()


test_promote_if_better(promote_if_better)

<details>
<summary>Hint</summary>

`client.get_model_version_by_alias(name, "champion")` gives you the champion's version and its `run_id`; `client.get_run(run_id).data.metrics` gives the metrics of a run. To tag a version, use `client.set_model_version_tag(name, version, key, value)`.
</details>

<details>
<summary>Solution</summary>

```python
def promote_if_better(candidate_run_id, metric="balanced_accuracy", min_gain=0.0, name="wine-good"):
    """Returns True if the candidate became the champion of `name`, False otherwise."""
    champion_version = client.get_model_version_by_alias(name, "champion")
    old = client.get_run(champion_version.run_id).data.metrics[metric]
    new = client.get_run(candidate_run_id).data.metrics[metric]
    if new - old <= min_gain:
        return False
    version = mlflow.register_model(f"runs:/{candidate_run_id}/model", name)
    client.set_registered_model_alias(name, "champion", version.version)
    client.set_model_version_tag(name, version.version, "promoted_because",
                                 f"{metric} {new:.3f} against {old:.3f} for version {champion_version.version}, margin {min_gain:.3f}")
    return True
```
</details>

Now apply it to the best run of the sweep, with a margin of two standard deviations of the split noise measured above.

In [ ]:
best = sweep.iloc[0]
promoted = promote_if_better(best.run_id, "balanced_accuracy", min_gain=2 * split_scores.std())
champion_version = client.get_model_version_by_alias("wine-good", "champion")
print("promoted:", promoted, "| champion is now version", champion_version.version, champion_version.tags)

Balanced accuracy treats a missed good wine and a false alarm as equally bad. Suppose instead that a missed good wine costs 5 (the shop does not stock it) and a false alarm costs 1 (a mediocre wine on the shelf). The cell below computes that cost per wine for each sweep run, from the model each run logged, and adds it to the run as a new metric.

In [ ]:
FALSE_ALARM, MISSED_GOOD = 1, 5

costs = []
for run_id in sweep.run_id:
    out = mlflow.pyfunc.load_model(f"runs:/{run_id}/model").predict(X_test)
    false_alarms = int(((out["good"] == 1) & (y_test.values == 0)).sum())
    missed = int(((out["good"] == 0) & (y_test.values == 1)).sum())
    cost = (FALSE_ALARM * false_alarms + MISSED_GOOD * missed) / len(y_test)
    client.log_metric(run_id, "cost_per_wine", cost)
    costs.append(cost)
sweep["cost_per_wine"] = costs
sweep.drop(columns="run_id").sort_values("cost_per_wine").round(3)

You should find that the best sweep run is not promoted, and that the lowest cost is about 0.31 per wine.

<details>
<summary>Question - why was the best sweep run not promoted?</summary>

The best sweep run (no depth limit, threshold 0.3) is the same model as the current champion, trained the same way on the same split: 0.830 against 0.830, a gain of zero, and the margin was 2 × 0.011 = 0.023. Training a model again is not an improvement. Note also the spread over five splits, 0.809 to 0.838: the gap between thresholds 0.3 and 0.4 in the sweep (0.830 against 0.807) is about two standard deviations, on the edge of what you can trust.
</details>

<details>
<summary>Question - is the run with the lowest cost the one with the best balanced accuracy?</summary>

Here yes (0.308 per wine), and the order of the other runs is the same. With misses five times as costly as false alarms, low thresholds win, and balanced accuracy happens to agree. Swap the costs (a false alarm costs 5, a miss costs 1) and run the cell again: the threshold-0.3 model goes from best to second worst (0.618 per wine) and the unlimited-depth model at threshold 0.5 wins (0.237). The metric is a business decision. Write it down before the sweep, not after looking at the results.
</details>

<details>
<summary>Question - <code>promote_if_better</code> assumes that higher is better. How would you use it with <code>cost_per_wine</code>?</summary>

Log the cost with its sign flipped (`-cost`) under another name, or give the function a `higher_is_better` argument that flips the comparison. Do not reuse a rule whose direction is wrong for the metric: it will promote the worst model and leave a tag that says why.
</details>

<details>
<summary>Question - who should be allowed to call this function?</summary>

Not whoever happens to have a notebook open. Usually a scheduled job or a CI step runs it after training, with the same test set and the same rule every time; a person reviews the reason in the tag. A common refinement is to move a `challenger` alias first, compare the two on live traffic, and only then move `champion`. Every promotion leaves a record, and can be undone by moving the alias back.
</details>

# 9️⃣ After deployment (optional)

> ##### Learning Objectives
>
> * See why accuracy can stay high while a model fails
> * Detect a change in the inputs without labels

Suppose the first version of the model was trained on white wines only, because that is all the first client sold. It worked well. Then the client starts selling red wines too, and nobody tells the team that maintains the model.

Here is that first model, scored on new white wines and on the red ones.

In [ ]:
white_train = X_train["type"] == "white"
first_model = WineModel(RandomForestClassifier(**params)).fit(X_train[white_train], y_train[white_train])

batches = {"new white wines": X_test["type"] == "white", "red wines": X_test["type"] == "red"}
rows = []
for batch, mask in batches.items():
    out = first_model.predict(None, X_test[mask])
    rows.append({"batch": batch, "wines": int(mask.sum()), "predicted good": out["good"].mean(), **metrics(y_test[mask], out)})
pd.DataFrame(rows).set_index("batch").round(3)

Look at accuracy first, then at recall and at the share of wines predicted good.

This table needs the true labels, and in production you do not have them yet: you learn which wines were good weeks later, if ever. What you do have, immediately, is the inputs and the predictions. A first check is to compare each column of the incoming batch with the data the model was trained on.

### Exercise - implement `drift_report`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
>
> You should spend up to 10 minutes on this exercise.
> ```

For each numeric column (all but `type`), compute how far the mean of the incoming batch is from the mean of the reference data, in units of the reference's standard deviation: `(incoming mean - reference mean) / reference std`. Return a pandas Series indexed by column, sorted by the size of the shift, largest first.

In [ ]:
def drift_report(reference: pd.DataFrame, incoming: pd.DataFrame) -> pd.Series:
    """Returns the shift of each numeric column, in reference standard deviations, largest first."""
    raise NotImplementedError()


test_drift_report(drift_report)

<details>
<summary>Solution</summary>

```python
def drift_report(reference: pd.DataFrame, incoming: pd.DataFrame) -> pd.Series:
    """Returns the shift of each numeric column, in reference standard deviations, largest first."""
    ref, new = reference.drop(columns="type"), incoming.drop(columns="type")
    return ((new.mean() - ref.mean()) / ref.std()).sort_values(key=abs, ascending=False)
```
</details>

Now the two batches against the training data of the first model. The red batch is also logged as a run of its own: monitoring results are just more runs, and the UI can plot them over time.

In [ ]:
reference = X_train[white_train]
report = pd.DataFrame({batch: drift_report(reference, X_test[mask]) for batch, mask in batches.items()})
report = report.sort_values("red wines", key=abs, ascending=False)

with mlflow.start_run(run_name="monitoring-red-batch", tags={"monitoring": "drift"}):
    mlflow.log_metrics({f"shift_{c.replace(' ', '_')}": v for c, v in report["red wines"].items()})
    mlflow.log_metric("predicted_good_share", first_model.predict(None, X_test[batches["red wines"]])["good"].mean())

report.round(2)

You should find shifts of up to about 2.5 standard deviations for the red wines, and below 0.1 for the new white ones.

<details>
<summary>Question - would accuracy alone have warned you? What would?</summary>

No. Accuracy is 0.852 on the red wines against 0.881 on the new white ones, a drop nobody would worry about. It stays high because 85% of red wines are not good, and the model says "not good" to almost all of them. Recall shows the problem: the model finds 3% of the good red wines. Without labels, the share of wines predicted good would have warned you: it falls from 16.5% to 0.6%.
</details>

<details>
<summary>Question - which alert rule would you pick, and would it have fired on the new white wines?</summary>

For example: alert when any column moves by more than one standard deviation. The red batch has six such columns, the largest volatile acidity at 2.5; the new white wines never exceed 0.1. Here any threshold between 0.1 and 1 separates the two. In practice you set it from the spread of past batches that were fine, and smaller batches need a looser threshold, because their means move more by chance.
</details>

<details>
<summary>Question - does it matter, for the alert, whether you can explain the shift?</summary>

Red and white wines are made differently (white wines usually get more sulfur dioxide, for example), so the large shifts are not surprising to a winemaker. Notice that alcohol, the column that matters most for quality, barely moved (-0.08): the model fails for reasons that a check on its most important feature would miss. The alert does not need an explanation to be right. The explanation matters for what you do next.
</details>

<details>
<summary>Question - the alert fired. What do you do next?</summary>

First find out what changed, from the client: this is a new kind of wine, not a slow drift. Note also that the model never saw a red wine, so the `type` column was constant in its training data and it learnt nothing from it. Options: retrain with labelled red wines (the data set has 1,599), refuse red wines or send them to a person until then, or keep scoring them with a warning. To choose you need to know how many red wines arrive, what a wrong answer costs, and whether labels for them exist or can be bought.
</details>

# 🔟 Bonus

No solutions here. Pick what interests you.

* **Other models.** Try another estimator, or a `Pipeline` with scaling, through `run_experiment`, and compare the runs in the UI.
* **Fewer features.** Train on a subset of the columns. How much do you lose, and what does the signature become?
* **Quality as a number.** Predict `quality` as a regression instead of `good`. Which metrics make sense then, and what has to change in `WineModel`?
* **Duplicates.** 1,177 of the 6,497 wines are exact duplicates, and 474 of the 1,950 test rows also appear in the training set. Drop the duplicates before splitting, run a few experiments again, and compare. Log the data with `mlflow.data.from_pandas` and `mlflow.log_input`, so that runs on the old and the new data can be told apart in the UI.
* **Refusing bad rows.** Add checks to `WineModel.predict` for the cases of section 4️⃣ (an unknown `type`, a missing value), log the model again, and send it the broken rows in Python and over HTTP.

## Cleanup

Stops the model server and the tracking server.

In [ ]:
serving.terminate()
server.terminate()